In [1]:
import json
import pickle
import os

from pdgs_generation import *
import copy
from FPstatic import print_node
from information import *
from FPanalysis import *
from pathlib import Path
import jsbeautifier
from condition import *

from multiprocessing import Pool

In [14]:
contentEmbedd = 'HTMLDocument.createElement'

### JS Diff Behavior

Following are the ways in which we analyze the differentiating/discriminating/distinctive JS behavior:

1. Fingerprinting API calls (What are the conditions upon which these distinct calls are made? Did device information play a role in it?)
2. Cookies (What is the distribution of cookie calls for android and mobile? We only need to consider common scripts here.)
3. iFrames and tracker pixels (Is the code also embedding tracking content like iframes and pixels in the platform differentially?)

4. Other scripts that are embedded through javascript

In [2]:
def get_node_by_offset(graph, offset):
    # depth first search
    def dfs(node):
        if 'start' in node.get_attributes() and node.get_attributes()['start'] == offset:
            return node
        for child in node.get_children():
            n = dfs(child)
            if n:
                return n
        return None
    return dfs(graph)

def loadCode(filename):
    # save the code in a string
    with open(filename, 'r') as file:
        code = file.read()
    return code



#### Condition and iFlow

In [10]:
def read_stats(jsonfilename):
    a = {
        "desktop": {
            "unique_calls": 0,
            "conditional_nodes_counts": 0,
            "iflow_chain_counts": 0
        },
        "mobile": {
            "unique_calls": 0,
            "conditional_nodes_counts": 0,
            "iflow_chain_counts": 0
        }
    }
    with open(jsonfilename) as fp:
        d = json.load(fp)
        fp.close()
    
    for url in d:
        for platform in d[url]:
            if platform == 'code':
                continue
            for api in d[url][platform]:
                if len(d[url][platform][api]['iflow']) > 0:
                    a[platform]['iflow_chain_counts'] += 1
                # elif url == 'https://bat.bing.com/bat.js' and api == '13463,Navigator.userAgentData':
                #     a[platform]['iflow_chain_counts'] += 1
                a[platform]['unique_calls'] += 1
    return a

In [11]:
resultfiles = os.listdir('results')
desktop_iflows, mobile_iflows = 0,0
desktop_calls, mobile_calls = 0,0

for f in resultfiles:
    path = os.path.join('results',f)
    a = read_stats(path)
    desktop_calls += a['desktop']['unique_calls']
    mobile_calls += a['mobile']['unique_calls']
    desktop_iflows += a['desktop']['iflow_chain_counts']
    mobile_iflows += a['mobile']['iflow_chain_counts']

print("Desktop calls: ", desktop_calls)
print("Mobile calls: ", mobile_calls)
print("Desktop iflows: ", desktop_iflows, (desktop_iflows/desktop_calls)*100)
print("Mobile iflows: ", mobile_iflows, (mobile_iflows/mobile_calls)*100)


"""
Stats when Bing's script is fixed for iFlow
Desktop calls:  928
Mobile calls:  499
Desktop iflows:  141 15.193965517241379
Mobile iflows:  35 7.014028056112225
"""


Desktop calls:  928
Mobile calls:  499
Desktop iflows:  114 12.284482758620689
Mobile iflows:  35 7.014028056112225


"\nStats when Bing's script is fixed for iFlow\nDesktop calls:  928\nMobile calls:  499\nDesktop iflows:  141 15.193965517241379\nMobile iflows:  35 7.014028056112225\n"

#### Cookies

We find unique calls for setting and getting cookie values

Generally, the reasons for discrepencies in these calls are following:
1. A condition on the state of the cookie document (existence of a key value (etc. DedeUserID), RegEx filter, length of cookies available)
2. The function in which the cookie call is made was never called in either of the two platforms

In [18]:
cookieDir = './cookieData'
files = os.listdir(cookieDir)
mobile,desktop = 0,0
for f in files:
    filename = os.path.join(cookieDir, f)
    with open(filename, 'r') as file:
        data = json.load(file)
    for script in data:
        # print(data[script]['desktop']['cookie'])
        # break
        try:
            mobile += len(data[script]['mobile']['cookie'])
            desktop += len(data[script]['desktop']['cookie'])
        except:
            print(f)
        
print(f'mobile set cookies {mobile}')
print(f'desktop set cookies {desktop}')

mobile set cookies 239
desktop set cookies 276


#### iFrame/tracker pixels

Catching this is actually tough and even harder to justify. Mobile and Desktops are naturally different ecosystems and on the web the content that is rendered on them need not to be necessarily the same. iFrames can differ. To identify the embedded content that has the purpose of tracking the user needs some heuristics

First, API pattern recognition. Here is a verified example of a sequence of APIs through which tracker pixel injected into the device.

"18663,HTMLDocument.createElement",

"18689,HTMLImageElement.setAttribute",

"18773,HTMLImageElement.id",

"18819,HTMLDivElement.appendChild",

"19065,HTMLImageElement.width",

"19076,HTMLImageElement.height",

"19126,HTMLImageElement.setAttribute",

"19218,HTMLImageElement.setAttribute"

This is how it would look like in vv8 logs:

c197836:%createElement:{772312,HTMLDocument}:"iframe"
g198007:{65545,HTMLIFrameElement}:"style"
s198026:{666206,CSSStyleDeclaration}:"width":0
g198007:{65545,HTMLIFrameElement}:"style"
s198026:{666206,CSSStyleDeclaration}:"height":0
g198007:{65545,HTMLIFrameElement}:"style"
s198026:{666206,CSSStyleDeclaration}:"border":"none"
g198007:{65545,HTMLIFrameElement}:"style"
s198026:{666206,CSSStyleDeclaration}:"position":"absolute"
g198007:{65545,HTMLIFrameElement}:"style"
s198026:{666206,CSSStyleDeclaration}:"left":"-9999px"
g198007:{65545,HTMLIFrameElement}:"style"
s198026:{666206,CSSStyleDeclaration}:"top":"-9999px;"
g198007:{65545,HTMLIFrameElement}:"style"
s198026:{666206,CSSStyleDeclaration}:"overflow":"hidden"
g198035:{65545,HTMLIFrameElement}:"setAttribute"
c198035:%setAttribute:{65545,HTMLIFrameElement}:"width":0
g198062:{65545,HTMLIFrameElement}:"setAttribute"
c198062:%setAttribute:{65545,HTMLIFrameElement}:"height":0
g198090:{65545,HTMLIFrameElement}:"setAttribute"
c198090:%setAttribute:{65545,HTMLIFrameElement}:"src":"https\://cmp.osano.com"

The next step is to determine if it is a tracker pixel:
1. Determine that the source url belongs to a tracking domain
2. Determine that its dimensions are either 0x0 or 1x1

In [16]:
cookieDir = './iframeData'
files = os.listdir(cookieDir)
mobile,desktop = 0,0
for f in files:
    filename = os.path.join(cookieDir, f)
    with open(filename, 'r') as file:
        data = json.load(file)
    for script in data:
        # print(data[script]['desktop']['cookie'])
        # break
        try:
            mobile += len(data[script]['mobile'][contentEmbedd])
            desktop += len(data[script]['desktop'][contentEmbedd])
        except:
            print(f)
        
print(mobile, desktop)

1487 847


Conclusion (so far), Next tasks and Future goals

So far, we have been able to show:

a. Fingerprinting APIs are distinctly used for platforms in commonly executed JavaScript resources

b. We can trace device information for some of these APIs (at ~15% so far)

c. We have identified cases where theoretically we do not need to derive an iFlow on a conditional node because the one of the sources is in fact the sink. Accounting for this class of cases, our iFlow failed cases reduce significantly (we are yet to find out exactly how much)

d. Common scripts show almost similar cookie setting/getting behavior

e. iFrames/trackers disproportionaility is more significant - we need a vv8 postprocesser that can tie tracker information together
    Information we need: content/images that are embedded, src for imgs, dimensions, parent frame/subdocument embedding the content, src for the parent frame

Next task: 
1. Completing point c - what is the new iflow success rate?
2. We also need to track document.createElement("script") distinct calls in common scripts and trace the child scripts. We have done this before but we haven't really thought out what to do with such child scripts. Like iFrames, justifying the analysis of such child scripts from a tracking PoV will be hard because researchers will say that platforms can have their own specific scripts. But if we want to include this in our analysis, our argument should be along the lines of - if a specific script collects more data from one platform, then isnt that a concern?
3. A postprocessor for the iFrames
